# Dataset qualification: bound census and diagnostic receipts

This portable report consumes completed qualification artifacts for the pinned PASSAGE release. It does not launch source downloads, metadata scans, exact solvers, training or inference. Set `qualification_root` to an artifact bundle containing `source-authentication/`, `metadata/` and `route-audit/`. With the installed PASSAGE package, no private checkout is required. If that bundle is absent, the report displays `not_run` prerequisites and renders an explicit pending panel.

The complete core metadata census and the 196-scene stratified route roster have distinct denominators. The roster requests 1,176 objective/obstacle variants, each with a fixed 100,000-expansion exact-reference budget and strict tolerances. Failed and censored findings remain visible. Unequal selection probabilities prevent interpreting these diagnostic counts as population defect prevalence or confidence intervals. Recorded rectangle overlap and tile support do not certify strict source-cell independence.


In [ ]:
# Papermill overrides this cell before paths are resolved.
resolution = 256  # Export name only: this report presents all seven resolutions.
project_path = None
qualification_root = None  # Local or downloaded qualification artifact bundle; never an inferred dataset path.
dataset_path = None
dataset_repo = None  # Optional expected source repository; when set, it must match the artifacts.
dataset_revision = None  # Optional expected 40-hex source revision; when set, it must match.
output_dir = "outputs/notebook-reports"
result_root = None


In [ ]:
from pathlib import Path
import sys
import json

project_root = Path(project_path).resolve() if project_path else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
if (project_root / "src/passage").is_dir():
    sys.path.insert(0, str(project_root / "src"))
from passage.qualification_reports import qualification_report, render_qualification_figures, with_report_figures
from passage.notebook_reports import resolve_path, resolve_evidence_path, write_report, show_records
from IPython.display import display, Markdown, Image
artifact_path = resolve_evidence_path(qualification_root, project_root)
output_path = resolve_path(output_dir, project_root)
report = qualification_report(artifact_path)
expected_source = {"repo": dataset_repo, "revision": dataset_revision}
if any(value is not None and report["source"][key] != value for key, value in expected_source.items()):
    raise ValueError("Configured dataset identity differs from the qualification artifact source.")
section_statuses = {
    "source_authentication": "admitted" if report["completion"]["source_authentication"] else "not_run",
    "core_census": "admitted" if report["completion"]["metadata_census"] else "not_run",
    "qualification": "admitted" if report["completion"]["full_campaign"] else "not_run",
    "classical_diagnostics": "admitted" if report["completion"]["route_audit"] else "not_run",
    "geographic_support": "not_run",
}
display(Markdown(f"Qualification status: **{report['status']}**. {report['interpretation']}"))
if report["missing_prerequisites"]:
    show_records([{"prerequisite": p, "status": "not_run"} for p in report["missing_prerequisites"]])


In [ ]:
show_records([{
    "resolution": item["resolution"], "full_metadata_rows": item["metadata"]["population_rows"],
    "selected_scenes": item["routes"]["selected_scenes"],
    "requested_variants": item["routes"]["requested_variants"],
    **{key: item["routes"].get(key) for key in ("task_records", "legal", "stored_cost_agreement",
        "reference_exact", "reference_censored", "reference_no_path", "reference_not_run", "qualified")},
} for item in report["resolutions"]])
show_records([{"resolution": item["resolution"], "split": split, "rows": cell["rows"],
              "duplicate_id_rows": cell.get("duplicate_id_rows"), "missing_fields": cell.get("missing"),
              "invalid_fields": cell.get("invalid"), "footprint_statuses": cell.get("footprints")}
             for item in report["resolutions"] for split, cell in item["metadata"]["cells"].items()])
show_records([{"resolution": item["resolution"], "status": item["geography"].get("status"),
              "source_cell_windows": item["geography"].get("source_cell_windows", "unknown"),
              "split_pairs": item["geography"].get("split_pairs", {})} for item in report["resolutions"]])


In [ ]:
figure_outputs = render_qualification_figures(report, output_path)
report = with_report_figures(report, figure_outputs, output_path)
display(Image(filename=str(output_path / "fig_qualification.png")))
if "fig_qualification_metadata.png" in figure_outputs:
    display(Image(filename=str(output_path / "fig_qualification_metadata.png")))
if "fig_qualification_tile_support.png" in figure_outputs:
    display(Image(filename=str(output_path / "fig_qualification_tile_support.png")))
# The derived report copy binds figure paths and hashes without mutating the input report.
figure_receipt = {"report_sha256": report["binding"]["report_sha256"], "outputs": figure_outputs}
write_report(figure_receipt, output_path, "qualification_figures.json")
report_path = write_report(report, output_path, f"dataset_qualification-{resolution}.json")
display(Markdown(f"Bound report receipt: `{report_path}`. Planned counts remain separate from completed findings."))
